# Predictive Maintenance - Monitoring and Drift Detection

This notebook demonstrates drift detection and model monitoring capabilities.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from predictive_maintenance.data_generator import SyntheticDataGenerator
from predictive_maintenance.monitoring.drift_detection import DataDriftDetector, ModelPerformanceMonitor
from predictive_maintenance.models.classical_models import RULPredictor

%matplotlib inline
sns.set_style('whitegrid')

## 1. Generate Reference and Current Data

In [ ]:
# Generate reference data (training distribution)
generator = SyntheticDataGenerator(random_state=42)
reference_df, _ = generator.generate_complete_dataset(n_units=30)

# Generate current data (potentially drifted)
current_generator = SyntheticDataGenerator(random_state=123)
current_df, _ = current_generator.generate_complete_dataset(n_units=20)

print(f"Reference data: {reference_df.shape}")
print(f"Current data: {current_df.shape}")

## 2. Initialize Drift Detector

In [ ]:
# Initialize and fit drift detector on reference data
drift_detector = DataDriftDetector(threshold=0.05)
drift_detector.fit(reference_df)

print("Drift detector fitted on reference data")

## 3. Detect Data Drift

In [ ]:
# Detect drift in current data
drift_report = drift_detector.detect_drift(current_df)

print(f"Drift detected: {drift_report.has_drift}")
print(f"Drift score: {drift_report.drift_score:.2%}")
print(f"\nDrifted features: {drift_report.drifted_features}")

## 4. Visualize Drift Statistics

In [ ]:
# Extract p-values for visualization
features = []
p_values = []

for feature, stats in drift_report.statistics.items():
    features.append(feature)
    p_values.append(stats['p_value'])

# Create bar plot
plt.figure(figsize=(12, 6))
colors = ['red' if p < 0.05 else 'green' for p in p_values]
plt.bar(range(len(features)), p_values, color=colors)
plt.axhline(y=0.05, color='black', linestyle='--', label='Threshold (p=0.05)')
plt.xlabel('Feature')
plt.ylabel('P-value')
plt.title('Drift Detection - P-values by Feature')
plt.xticks(range(len(features)), features, rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Model Performance Monitoring

In [ ]:
# Initialize performance monitor
monitor = ModelPerformanceMonitor()

# Simulate logging metrics over time
for i in range(20):
    # Simulate degrading performance
    rmse = 20 + i * 0.5 + np.random.randn()
    mae = 15 + i * 0.3 + np.random.randn()
    r2 = 0.85 - i * 0.01 + np.random.randn() * 0.01
    
    monitor.log_metrics({
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    })

print("Logged 20 metric entries")

## 6. Visualize Performance Over Time

In [ ]:
# Get metrics dataframe
metrics_df = monitor.get_metrics_dataframe()

# Plot metrics over time
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(metrics_df.index, metrics_df['rmse'], marker='o')
axes[0].set_ylabel('RMSE')
axes[0].set_title('Model Performance - RMSE Over Time')
axes[0].grid(True)

axes[1].plot(metrics_df.index, metrics_df['mae'], marker='o', color='orange')
axes[1].set_ylabel('MAE')
axes[1].set_title('Model Performance - MAE Over Time')
axes[1].grid(True)

axes[2].plot(metrics_df.index, metrics_df['r2'], marker='o', color='green')
axes[2].set_ylabel('R²')
axes[2].set_xlabel('Sample Index')
axes[2].set_title('Model Performance - R² Over Time')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 7. Detect Performance Degradation

In [ ]:
# Check for performance degradation
degraded = monitor.detect_performance_degradation('r2', threshold=0.1)

if degraded:
    print("⚠️  Performance degradation detected!")
else:
    print("✓ Performance is stable")

# Get summary statistics
summary = monitor.get_summary_statistics()
print("\nSummary Statistics:")
for metric, stats in summary.items():
    print(f"\n{metric}:")
    for stat_name, value in stats.items():
        if value is not None:
            print(f"  {stat_name}: {value:.4f}")

## 8. Compare Sensor Distributions

In [ ]:
# Compare distributions of a specific sensor
sensor_col = 'sensor_1'

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(reference_df[sensor_col], bins=50, alpha=0.7, label='Reference', color='blue')
plt.hist(current_df[sensor_col], bins=50, alpha=0.7, label='Current', color='red')
plt.xlabel(sensor_col)
plt.ylabel('Frequency')
plt.title(f'{sensor_col} Distribution Comparison')
plt.legend()

plt.subplot(1, 2, 2)
plt.boxplot([reference_df[sensor_col], current_df[sensor_col]], 
            labels=['Reference', 'Current'])
plt.ylabel(sensor_col)
plt.title(f'{sensor_col} Box Plot Comparison')

plt.tight_layout()
plt.show()